# Tagged units vs. SI/MA population waveforms

Using the two artifacts already saved to disk:

1. the **big waveform/feature CSV** (`all_sessions_waveform_features.csv`, built
   by `waveform_features_dataset.ipynb`), and
2. the **opto-tagging metric CSVs** (`*_laser_response_metrics.csv`, from
   `optotagging_Anna_nwb_batch.ipynb`),

this notebook selects the **SI + MA** (ventral pallidum) units and the
**opto-tagged** units, then overlays their waveforms and features in the same
figures so you can see whether the tagged units stand out from the SI/MA
population.

No NWB re-reading is needed — everything comes from the two CSV sources.


## 1. Imports & configuration

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
import re
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from waveform_clustering import (
    FEATURE_COLS,
    load_features_dataset,
    normalize_waveform,
    clean_waveform_mask,
)

%matplotlib inline

# ---- Configuration -------------------------------------------------------
# Big per-unit dataset (all sessions, all units) from waveform_features_dataset.ipynb
DATASET_CSV = Path("/root/capsule/scratch/waveform_clustering/all_sessions_waveform_features.csv")

# Folder holding the opto-tagging metric CSVs (*_laser_response_metrics.csv)
OPTO_DIR = Path("/root/capsule/scratch/opto_tagging_Anna")

# Ventral pallidum is covered by the CCF acronyms SI + MA.
TARGET_REGIONS = ["SI", "MA"]

# Opto-tagging thresholds (same as the batch notebook).
RED_MIN_SIG_PULSES = 4     # red: >= 4 significant pulses
BLUE_MIN_SIG_PULSES = 5    # blue: >= 5 significant pulses
MAX_JITTER = 0.01          # s
MAX_ISI = 0.5              # pre-stim ISI-violation ratio

# Restrict which opto-tag types to include (optional). Set to None to keep all
# tagged units, or a list of tag types, e.g. ["external_blue"] for blue only.
TAG_TYPES = ["external_blue"]

# ---- Noise / artifact filtering ------------------------------------------
# Drop oscillatory, non-spike waveforms (many peaks, non-decaying baseline).
# Set FILTER_NOISE = False to disable, or loosen/tighten the thresholds.
FILTER_NOISE = True
NOISE_MAX_PEAKS = 4            # max prominent excursions in |waveform|
NOISE_MAX_ZERO_CROSSINGS = 8   # max baseline-crossings (oscillation count)
NOISE_MAX_LATE_ENERGY = 0.6    # max fraction of energy outside +/- core window
NOISE_CORE_MS = 0.7            # half-width (ms) of the "spike core" window

feature_cols = FEATURE_COLS

## 2. Load the big waveform / feature dataset

Reload the flat CSV back into a feature/metadata table plus a waveform matrix,
then keep only the **SI / MA** units.


In [ ]:
loaded = load_features_dataset(DATASET_CSV)
features = loaded["features"]
waveforms = loaded["waveforms"]
time_ms = loaded["time_ms"]

# Amplitude-normalize every stored (baseline-corrected, trough-aligned) waveform.
norm_waveforms = (
    np.vstack([normalize_waveform(w) for w in waveforms]) if len(waveforms) else waveforms
)

# QC filter applied to BOTH the SI/MA population and the tagged units below.
qc_mask = features["qc_pass"].astype(bool).values

# Noise filter: drop oscillatory / non-spike waveforms (applied to both groups).
if FILTER_NOISE:
    clean_mask, noise_metrics = clean_waveform_mask(
        norm_waveforms,
        time_ms,
        max_peaks=NOISE_MAX_PEAKS,
        max_zero_crossings=NOISE_MAX_ZERO_CROSSINGS,
        max_late_energy=NOISE_MAX_LATE_ENERGY,
        core_ms=NOISE_CORE_MS,
        return_metrics=True,
    )
    print(f"Noise filter: kept {int(clean_mask.sum())} / {len(clean_mask)} waveforms "
          f"({int((~clean_mask).sum())} flagged as noise).")
else:
    clean_mask = np.ones(len(features), dtype=bool)
    print("Noise filter disabled.")

# SI/MA units that also pass QC and the noise filter.
region_mask = features["region"].isin(TARGET_REGIONS).values & qc_mask & clean_mask
print(f"Total units in dataset: {len(features)}")
print(f"QC-passing units: {int(qc_mask.sum())}")
print(f"SI/MA units (QC + non-noise): {int(region_mask.sum())}")
print(features.loc[region_mask, "region"].value_counts())

In [ ]:
# Diagnostic: inspect what the noise filter flags (tune thresholds if needed).
if FILTER_NOISE:
    noisy_idx = np.where(~clean_mask)[0]
    clean_idx = np.where(clean_mask)[0]
    rng = np.random.default_rng(0)
    show_noisy = rng.choice(noisy_idx, size=min(40, len(noisy_idx)), replace=False)
    show_clean = rng.choice(clean_idx, size=min(40, len(clean_idx)), replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for i in show_clean:
        axes[0].plot(time_ms, norm_waveforms[i], color="tab:green", linewidth=0.6, alpha=0.6)
    axes[0].set_title(f"Kept (clean) - sample of {len(clean_idx)}")
    for i in show_noisy:
        axes[1].plot(time_ms, norm_waveforms[i], color="tab:red", linewidth=0.6, alpha=0.6)
    axes[1].set_title(f"Flagged as noise - sample of {len(noisy_idx)}")
    for ax in axes:
        ax.set_xlabel("Time (ms)")
    axes[0].set_ylabel("Normalized amplitude")
    plt.tight_layout()
    plt.show()
else:
    print("Noise filter disabled; nothing to inspect.")

## 3. Identify the opto-tagged units

Read each `*_laser_response_metrics.csv`, apply the same tagging query used by
the batch notebook (red: >= `RED_MIN_SIG_PULSES` significant pulses; blue: >=
`BLUE_MIN_SIG_PULSES`, excluding units already red-tagged), and collect the
`(session, unit_index)` pairs. The session core (e.g. `839480_2026-06-03_...`)
is parsed from the filename so it matches the dataset's `session_name`.


In [ ]:
SESSION_CORE_RE = re.compile(r"(\d+_\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})")


def tagged_from_metrics(metrics, trial_type, min_sig_pulses, max_jitter, max_isi):
    """Rows of `metrics` passing the tagging criteria for one trial type."""
    q = []
    col_pulses = f"{trial_type}_train_max_num_sig_pulses"
    col_jitter = f"{trial_type}_train_best_mean_jitter"
    if col_pulses in metrics.columns:
        q.append(f"{col_pulses} >= {min_sig_pulses}")
    if col_jitter in metrics.columns:
        q.append(f"{col_jitter} < {max_jitter}")
    if "pre_stim_isi_ratio" in metrics.columns:
        q.append(f"pre_stim_isi_ratio < {max_isi}")
    if not q:
        return metrics.iloc[0:0]
    return metrics.query(" and ".join(q))


def load_tagged_units(opto_dir, red_min_sig_pulses, blue_min_sig_pulses,
                      max_jitter, max_isi):
    """Collect tagged (session_name, unit_index, tag_type) from metric CSVs."""
    rows = []
    csvs = sorted(Path(opto_dir).glob("*_laser_response_metrics.csv"))
    print(f"Found {len(csvs)} metric CSV(s) in {opto_dir}")
    for csv in csvs:
        m = SESSION_CORE_RE.search(csv.name)
        if not m:
            print(f"  [skip] cannot parse session from {csv.name}")
            continue
        session_core = m.group(1)
        metrics = pd.read_csv(csv)
        if "unit_id" not in metrics.columns:
            continue
        trial_types = sorted({
            c[: -len("_train_max_num_sig_pulses")]
            for c in metrics.columns if c.endswith("_train_max_num_sig_pulses")
        })
        red_types = [t for t in trial_types if "red" in t]
        blue_types = [t for t in trial_types if "blue" in t]

        red_idx = set()
        for tt in red_types:
            tg = tagged_from_metrics(metrics, tt, red_min_sig_pulses, max_jitter, max_isi)
            red_idx.update(tg.index.tolist())
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))
        for tt in blue_types:
            tg = tagged_from_metrics(metrics, tt, blue_min_sig_pulses, max_jitter, max_isi)
            tg = tg[~tg.index.isin(red_idx)]
            for uid in tg["unit_id"].astype(int):
                rows.append((session_core, int(uid), tt))

    tagged = pd.DataFrame(rows, columns=["session_name", "unit_index", "tag_type"])
    return tagged.drop_duplicates(subset=["session_name", "unit_index"]).reset_index(drop=True)


tagged_df = load_tagged_units(
    OPTO_DIR,
    red_min_sig_pulses=RED_MIN_SIG_PULSES,
    blue_min_sig_pulses=BLUE_MIN_SIG_PULSES,
    max_jitter=MAX_JITTER,
    max_isi=MAX_ISI,
)
print(f"\nTagged units found: {len(tagged_df)}")
print(tagged_df["tag_type"].value_counts())
tagged_df.head()

## 4. Match tagged units into the dataset

Join the tagged `(session, unit)` pairs onto the big dataset so each tagged unit
picks up its stored waveform + features. Then build masks for the SI/MA
population and the tagged units.


In [ ]:
# Build a matching key on both sides.
def make_key(df):
    return (df["session_name"].astype(str) + "|" + df["unit_index"].astype(int).astype(str))

# Optionally restrict to specific tag types (e.g. external_blue only).
if TAG_TYPES is not None:
    tagged_sel = tagged_df[tagged_df["tag_type"].isin(TAG_TYPES)].copy()
    print(f"Restricting tagged units to tag types {TAG_TYPES}: "
          f"{len(tagged_sel)} / {len(tagged_df)} units")
else:
    tagged_sel = tagged_df.copy()
    print(f"Using all {len(tagged_sel)} tagged units (no tag-type filter)")

feat_key = make_key(features)
tagged_key = set(make_key(tagged_sel))

# Tagged units, restricted to those passing QC and the noise filter (same
# filters as the SI/MA population).
tagged_matched_any = feat_key.isin(tagged_key).values
tagged_mask = tagged_matched_any & qc_mask & clean_mask

n_tagged_matched = int(tagged_matched_any.sum())
n_tagged_qc = int((tagged_matched_any & qc_mask).sum())
n_tagged_clean = int(tagged_mask.sum())
n_tagged_in_sima = int((tagged_mask & region_mask).sum())
print(f"Tagged units matched in dataset: {n_tagged_matched} / {len(tagged_sel)}")
print(f"  passing QC: {n_tagged_qc}")
print(f"  passing QC + non-noise: {n_tagged_clean}")
print(f"  of those in SI/MA: {n_tagged_in_sima}")
print(f"SI/MA population units (QC + non-noise): {int(region_mask.sum())}")

if n_tagged_matched < len(tagged_sel):
    missing = len(tagged_sel) - n_tagged_matched
    print(f"\n[note] {missing} tagged unit(s) were not found in the dataset "
          "(their opto session may not be in the dataset, or the unit produced "
          "no valid waveform).")

# Region breakdown of the kept tagged units.
print("\nTagged (QC + non-noise) units by region:")
print(features.loc[tagged_mask, "region"].fillna("None").value_counts())

## 5. Overlay waveforms: SI/MA population vs. tagged units

Grey = SI/MA population; black = its mean. Coloured = tagged units; bold =
their mean. This is the direct visual test of whether the tagged units differ.


In [ ]:
pop_mask = region_mask                    # SI/MA population
tag_mask = tagged_mask                     # all matched tagged units

fig, ax = plt.subplots(figsize=(9, 6))

# SI/MA population
pop_wf = norm_waveforms[pop_mask]
for w in pop_wf:
    ax.plot(time_ms, w, color="lightgrey", linewidth=0.4, zorder=1)
if len(pop_wf):
    ax.plot(time_ms, pop_wf.mean(axis=0), color="black", linewidth=2.5,
            label=f"SI/MA mean (n={len(pop_wf)})", zorder=3)

# Tagged units
tag_wf = norm_waveforms[tag_mask]
for w in tag_wf:
    ax.plot(time_ms, w, color="tab:red", linewidth=0.7, alpha=0.5, zorder=2)
if len(tag_wf):
    ax.plot(time_ms, tag_wf.mean(axis=0), color="tab:red", linewidth=2.5,
            label=f"Tagged mean (n={len(tag_wf)})", zorder=4)

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Normalized amplitude")
ax.set_title("SI/MA population vs. opto-tagged units")
ax.legend()
plt.tight_layout()
plt.show()

### 5a. Compare only tagged units that lie in SI/MA

Restrict the tagged group to SI/MA so both groups are anatomically matched.


In [ ]:
tag_in_sima = tagged_mask & region_mask
pop_only = region_mask & ~tagged_mask     # SI/MA, untagged

fig, ax = plt.subplots(figsize=(9, 6))
pop_wf = norm_waveforms[pop_only]
for w in pop_wf:
    ax.plot(time_ms, w, color="lightgrey", linewidth=0.4, zorder=1)
if len(pop_wf):
    ax.plot(time_ms, pop_wf.mean(axis=0), color="black", linewidth=2.5,
            label=f"SI/MA untagged mean (n={len(pop_wf)})", zorder=3)

tag_wf = norm_waveforms[tag_in_sima]
for w in tag_wf:
    ax.plot(time_ms, w, color="tab:red", linewidth=1.0, alpha=0.7, zorder=2)
if len(tag_wf):
    ax.plot(time_ms, tag_wf.mean(axis=0), color="tab:red", linewidth=2.5,
            label=f"SI/MA tagged mean (n={len(tag_wf)})", zorder=4)

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Normalized amplitude")
ax.set_title("SI/MA: untagged vs. tagged units")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Compare waveform features

Overlay the two groups in feature space and compare the key separators
(trough-to-peak duration, half-width).


In [ ]:
# Feature scatter: SI/MA population vs tagged units
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.scatter(features.loc[pop_mask, "trough_to_peak_ms"],
           features.loc[pop_mask, "half_width_ms"],
           s=25, c="lightgrey", edgecolor="grey", linewidth=0.3,
           label=f"SI/MA (n={int(pop_mask.sum())})", zorder=1)
ax.scatter(features.loc[tag_mask, "trough_to_peak_ms"],
           features.loc[tag_mask, "half_width_ms"],
           s=90, c="tab:red", marker="*", edgecolor="black", linewidth=0.5,
           label=f"Tagged (n={int(tag_mask.sum())})", zorder=2)
ax.set_xlabel("Trough-to-peak duration (ms)")
ax.set_ylabel("Half-width (ms)")
ax.set_title("Waveform feature space: SI/MA vs. tagged")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distributions of key features (SI/MA population vs tagged), as overlaid
# density histograms so the two groups are directly comparable.
compare_cols = ["trough_to_peak_ms", "half_width_ms", "peak_trough_ratio", "repolarization_slope"]
# Optional per-feature x-axis limits (zoom in where the data is concentrated).
xlims = {"half_width_ms": (0, 0.4)}
fig, axes = plt.subplots(1, len(compare_cols), figsize=(4 * len(compare_cols), 4))
for ax, col in zip(axes, compare_cols):
    pop_vals = features.loc[pop_mask, col].dropna()
    tag_vals = features.loc[tag_mask, col].dropna()
    # Shared bins so both histograms line up. When an x-limit is set, compute
    # the bins within that range so the bin size matches the zoomed view.
    if col in xlims:
        bins = np.linspace(xlims[col][0], xlims[col][1], 31)
    else:
        all_vals = pd.concat([pop_vals, tag_vals])
        bins = np.histogram_bin_edges(all_vals, bins=30)
    ax.hist(pop_vals, bins=bins, density=True, color="lightgrey", alpha=0.8,
            label=f"SI/MA (n={len(pop_vals)})")
    ax.hist(tag_vals, bins=bins, density=True, color="tab:red", alpha=0.5,
            label=f"Tagged (n={len(tag_vals)})")
    # Dashed lines mark each group's mean.
    ax.axvline(pop_vals.mean(), color="black", linestyle="--", linewidth=1.2)
    ax.axvline(tag_vals.mean(), color="tab:red", linestyle="--", linewidth=1.8)
    ax.set_title(col)
    ax.set_xlabel(col)
    if col in xlims:
        ax.set_xlim(*xlims[col])
axes[0].set_ylabel("Density")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary: mean +/- std of each feature for the two groups.
summary = pd.DataFrame({
    "SI/MA_mean": features.loc[pop_mask, feature_cols].mean(),
    "SI/MA_std": features.loc[pop_mask, feature_cols].std(),
    "tagged_mean": features.loc[tag_mask, feature_cols].mean(),
    "tagged_std": features.loc[tag_mask, feature_cols].std(),
})
summary["n_SI/MA"] = int(pop_mask.sum())
summary["n_tagged"] = int(tag_mask.sum())
summary

## 7. PCA — SI/MA population vs. tagged units

Project both groups into a low-dimensional space to look for separation. We fit
PCA on the union of the two groups so the components capture their combined
variance, then colour by group. Two views are shown: PCA on the **morphology
features** and PCA directly on the **normalized waveform shapes**.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Two groups: SI/MA (untagged) vs tagged. Units that are both count as tagged.
combined_mask = region_mask | tagged_mask
is_tagged = tagged_mask[combined_mask]
group = np.where(is_tagged, "tagged", "SI/MA")

# --- PCA on the morphology features ---------------------------------------
Xf = features.loc[combined_mask, feature_cols].to_numpy()
Xf = np.nan_to_num(Xf, nan=0.0, posinf=0.0, neginf=0.0)
Xf_scaled = StandardScaler().fit_transform(Xf)
pca_f = PCA(n_components=2).fit(Xf_scaled)
Zf = pca_f.transform(Xf_scaled)

# --- PCA on the normalized waveform shapes --------------------------------
Xw = norm_waveforms[combined_mask]
pca_w = PCA(n_components=2).fit(Xw)
Zw = pca_w.transform(Xw)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, Z, pca, title in (
    (axes[0], Zf, pca_f, "PCA of morphology features"),
    (axes[1], Zw, pca_w, "PCA of waveform shapes"),
):
    m_pop = group == "SI/MA"
    m_tag = group == "tagged"
    ax.scatter(Z[m_pop, 0], Z[m_pop, 1], s=25, c="lightgrey", edgecolor="grey",
               linewidth=0.3, label=f"SI/MA (n={int(m_pop.sum())})", zorder=1)
    ax.scatter(Z[m_tag, 0], Z[m_tag, 1], s=90, c="tab:red", marker="*",
               edgecolor="black", linewidth=0.5,
               label=f"Tagged (n={int(m_tag.sum())})", zorder=2)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print("Feature PCA explained variance:", np.round(pca_f.explained_variance_ratio_, 3))
print("Waveform PCA explained variance:", np.round(pca_w.explained_variance_ratio_, 3))

In [ ]:
# 3D PCA (first 3 components) for both feature- and waveform-space views.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (enables 3D projection)

# Refit with 3 components on the same combined groups.
pca_f3 = PCA(n_components=3).fit(Xf_scaled)
Zf3 = pca_f3.transform(Xf_scaled)
pca_w3 = PCA(n_components=3).fit(Xw)
Zw3 = pca_w3.transform(Xw)

fig = plt.figure(figsize=(15, 6.5))
for pos, Z, pca, title in (
    (121, Zf3, pca_f3, "PCA of morphology features (3D)"),
    (122, Zw3, pca_w3, "PCA of waveform shapes (3D)"),
):
    ax = fig.add_subplot(pos, projection="3d")
    m_pop = group == "SI/MA"
    m_tag = group == "tagged"
    ax.scatter(Z[m_pop, 0], Z[m_pop, 1], Z[m_pop, 2], s=18, c="lightgrey",
               edgecolor="grey", linewidth=0.2, alpha=0.6,
               label=f"SI/MA (n={int(m_pop.sum())})")
    ax.scatter(Z[m_tag, 0], Z[m_tag, 1], Z[m_tag, 2], s=70, c="tab:red",
               marker="*", edgecolor="black", linewidth=0.4,
               label=f"Tagged (n={int(m_tag.sum())})")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.set_zlabel(f"PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print("Feature PCA (3D) explained variance:", np.round(pca_f3.explained_variance_ratio_, 3),
      f"| cumulative: {pca_f3.explained_variance_ratio_.sum()*100:.1f}%")
print("Waveform PCA (3D) explained variance:", np.round(pca_w3.explained_variance_ratio_, 3),
      f"| cumulative: {pca_w3.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# Interactive, rotatable 3D PCA (drag to rotate, scroll to zoom).
# Uses Plotly; if it's not installed run:  %pip install plotly
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

# Choose which space to view: "waveform" (Zw3) or "feature" (Zf3).
VIEW = "waveform"
Z3 = Zw3 if VIEW == "waveform" else Zf3
pca_sel = pca_w3 if VIEW == "waveform" else pca_f3

m_pop = group == "SI/MA"
m_tag = group == "tagged"

MARKER_SIZE = 4  # same size for both groups
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=Z3[m_pop, 0], y=Z3[m_pop, 1], z=Z3[m_pop, 2],
    mode="markers", name=f"SI/MA (n={int(m_pop.sum())})",
    marker=dict(size=MARKER_SIZE, color="royalblue", opacity=0.6),
))
fig.add_trace(go.Scatter3d(
    x=Z3[m_tag, 0], y=Z3[m_tag, 1], z=Z3[m_tag, 2],
    mode="markers", name=f"Tagged (n={int(m_tag.sum())})",
    marker=dict(size=MARKER_SIZE, color="crimson", opacity=0.85),
))
fig.update_layout(
    title=f"Interactive 3D PCA ({VIEW} space) - drag to rotate",
    scene=dict(
        xaxis_title=f"PC1 ({pca_sel.explained_variance_ratio_[0]*100:.1f}%)",
        yaxis_title=f"PC2 ({pca_sel.explained_variance_ratio_[1]*100:.1f}%)",
        zaxis_title=f"PC3 ({pca_sel.explained_variance_ratio_[2]*100:.1f}%)",
    ),
    width=850, height=700,
)
fig.show()

In [ ]:
# Distribution of each principal component, SI/MA vs tagged.
m_pop = group == "SI/MA"
m_tag = group == "tagged"

spaces = [
    ("Feature PCA", Zf3, pca_f3),
    ("Waveform PCA", Zw3, pca_w3),
]
n_pcs = 3
fig, axes = plt.subplots(len(spaces), n_pcs, figsize=(5 * n_pcs, 4 * len(spaces)))
axes = np.atleast_2d(axes)

for r, (name, Z, pca) in enumerate(spaces):
    for pc in range(n_pcs):
        ax = axes[r, pc]
        pop_vals = Z[m_pop, pc]
        tag_vals = Z[m_tag, pc]
        # Shared bins so the two distributions are directly comparable.
        bins = np.histogram_bin_edges(Z[:, pc], bins=30)
        ax.hist(pop_vals, bins=bins, density=True, color="lightgrey",
                alpha=0.8, label=f"SI/MA (n={m_pop.sum()})")
        ax.hist(tag_vals, bins=bins, density=True, color="tab:red",
                alpha=0.5, label=f"Tagged (n={m_tag.sum()})")
        ax.axvline(pop_vals.mean(), color="black", linestyle="--", linewidth=1)
        ax.axvline(tag_vals.mean(), color="tab:red", linestyle="--", linewidth=1.5)
        ax.set_title(f"{name} - PC{pc + 1} "
                     f"({pca.explained_variance_ratio_[pc]*100:.1f}% var)")
        ax.set_xlabel(f"PC{pc + 1}")
        if pc == 0:
            ax.set_ylabel("Density")
        if r == 0 and pc == 0:
            ax.legend()
plt.tight_layout()
plt.show()

## 7. Notes

- Grey = SI/MA population (from the big dataset CSV); red = opto-tagged units
  (matched from the `*_laser_response_metrics.csv` files by session + unit).
- Section 5 overlays tagged units on the whole SI/MA population; section 5a
  restricts the comparison to tagged units that are themselves in SI/MA.
- Adjust `RED_MIN_SIG_PULSES` / `BLUE_MIN_SIG_PULSES` / `MAX_JITTER` /
  `MAX_ISI` in the config cell to match the thresholds you used when tagging.
- If few/no tagged units match, check that the opto sessions are included in the
  big dataset and that `OPTO_DIR` points at the saved metric CSVs.
